# Bronze inventory
Connect to the platform MinIO S3 API as described in README.md. Use the inventory table to explain what landed, then open a collection to show the actual records.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from bronze_review import decode_file, inventory, open_bronze

bronze = open_bronze()
catalog = inventory(bronze)
print(f"Objects listed: {len(bronze.objects)}")

In [ ]:
columns = [
    "kind", "source", "collection", "key", "bytes", "rows", "status", "fields",
    "run_id", "published_at", "generation", "partition", "offset", "description",
]
inventory_df = pd.DataFrame.from_records(catalog, columns=columns)
inventory_df.to_csv(Path("bronze_inventory.csv"), index=False, encoding="utf-8-sig")

print("Dataset inventory")
display(inventory_df[["kind", "source", "collection", "rows", "status", "description"]])
exceptions_df = inventory_df[inventory_df["status"].fillna("ok").ne("ok")]
if not exceptions_df.empty:
    print("Inventory exceptions")
    display(exceptions_df[["key", "status"]])

Batch rows are counted per snapshot and should not be summed to derive unique business records. Kafka rows represent deliveries, each identified by generation, partition, and offset. Unclassified objects remain in MinIO and the inventory for manual inspection.

## Verify seeded bookings in Bronze

Run this check after the ERPNext REST and dealer SQL ingestion DAGs succeed. It checks both Sales Order snapshots for the same batch and shows up to 10 matching rows per source.

For `VOL001`, the seed log reported 100 ERPNext bookings. Set `expected_dealer_bookings` to `dealer.bookings` from the seed config. A blank expectation reports the count without checking it against a target. Leave the run IDs blank to use the latest successful snapshot for each source, or set them to the exact Airflow run IDs to compare a fixed pair of snapshots.

In [ ]:
batch_id = "VOL002"
expected_parent_bookings = 100
expected_dealer_bookings = None  # Set this to dealer.bookings in seed-volume.json.
parent_run_id = ""              # Optional exact ERPNext REST Airflow run ID.
dealer_run_id = ""              # Optional exact dealer SQL Airflow run ID.

batch_id = batch_id.strip().upper()
if not batch_id:
    raise ValueError("Enter the batch ID from the seed config, such as VOL002.")

booking_checks = []
booking_samples = {}
sources = (
    ("erpnext", "Sales Order", "custom_otc_seed_id", "P", expected_parent_bookings, parent_run_id),
    ("dealer-sql", "sales_orders", "id", "D", expected_dealer_bookings, dealer_run_id),
)

for booking_source, booking_collection, id_field, company_prefix, expected, run_id in sources:
    prefix = f"{company_prefix}-{batch_id}-"
    candidates = inventory_df.loc[
        inventory_df["kind"].eq("snapshot")
        & inventory_df["source"].eq(booking_source)
        & inventory_df["collection"].eq(booking_collection)
    ]
    if run_id.strip():
        candidates = candidates.loc[candidates["run_id"].eq(run_id.strip())]
    if candidates.empty:
        booking_checks.append({
            "source": booking_source, "run_id": run_id or None, "prefix": prefix,
            "expected": expected, "found": None, "status": "SNAPSHOT MISSING",
        })
        continue

    snapshot = candidates.sort_values(["published_at", "run_id", "key"]).iloc[-1]
    result = {
        "source": booking_source, "run_id": snapshot["run_id"], "prefix": prefix,
        "expected": expected, "found": None,
    }
    if snapshot["status"] != "ok":
        booking_checks.append({**result, "status": snapshot["status"]})
        continue

    booking_records, _ = decode_file(snapshot["key"], bronze.read(snapshot["key"]))
    if booking_records is None:
        booking_checks.append({**result, "status": "UNSUPPORTED SNAPSHOT"})
        continue
    if booking_records and not any(id_field in record for record in booking_records):
        booking_checks.append({**result, "status": f"MISSING FIELD: {id_field}"})
        continue

    matches = [
        record for record in booking_records
        if str(record.get(id_field, "")).startswith(prefix)
    ]
    count = len({record[id_field] for record in matches})
    if len(matches) != count:
        status = "DUPLICATE BOOKING IDS"
    elif expected is not None:
        status = "PASS" if count == expected else "COUNT MISMATCH"
    else:
        status = "COUNT ONLY" if count else "NO MATCHES"
    booking_checks.append({**result, "found": count, "status": status})

    if matches:
        sample_fields = (
            (id_field, "name", "customer", "transaction_date", "grand_total")
            if booking_source == "erpnext"
            else (id_field, "customer_id", "order_date")
        )
        booking_samples[booking_source] = pd.DataFrame.from_records(matches[:10]).reindex(
            columns=sample_fields
        )

display(pd.DataFrame(booking_checks).astype({"expected": "Int64", "found": "Int64"}))
for booking_source, sample in booking_samples.items():
    print(f"{booking_source} | first {len(sample)} matching bookings")
    display(sample)

## Open the actual records
The default example shows the latest dealer customer snapshot. Change the source and collection to show another snapshot, for example `erpnext` and `Sales Invoice`. For an inbound CSV, JSON file, or Kafka envelope, copy its full key from `inventory_df` or `bronze_inventory.csv` into `object_key`. One row represents one source record. Nested ERP line items stay with their parent record.

In [ ]:
source = "dealer-sql"
collection = "customers"
object_key = ""

if object_key:
    selected = inventory_df.loc[inventory_df["key"].eq(object_key)]
else:
    snapshots = inventory_df.loc[
        inventory_df["kind"].eq("snapshot") & inventory_df["source"].eq(source)
    ]
    latest_run = snapshots.sort_values(["published_at", "run_id"]).tail(1)["run_id"]
    selected = snapshots.loc[
        snapshots["run_id"].isin(latest_run) & snapshots["collection"].eq(collection)
    ]

if len(selected) != 1:
    raise ValueError("Select a source and collection from inventory_df, or enter one exact object key.")
entry = selected.iloc[0]
if entry["status"] != "ok":
    raise ValueError(f"Choose a readable data object. Selected status: {entry['status']}")
records, fields = decode_file(entry["key"], bronze.read(entry["key"]))
if records is None:
    raise ValueError("Choose a JSON, JSONL, or CSV object to display as a DataFrame.")

records_df = pd.json_normalize(records, max_level=1) if records else pd.DataFrame(columns=fields)
print(f"{entry['source']} | {entry['collection']} | {len(records_df)} records")
print(entry["description"])
display(records_df.head(20))